In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import polars as pl
import plotly.express as px
from utilsforecast.evaluation import evaluate
import plotly.io as pio

from utilsforecast.losses import *
from functools import partial
from plotting_utils import (
    plotly_series as plot_series,
)
import torch.nn as nn
import torch

# Introduction to Deep Learning for Time Series Forecasting

Time series forecasting is the process of predicting future values based on previously observed data points, where the data is ordered in time. This is crucial in many real-world applications, such as:

- **Energy demand prediction** (like our smart meter dataset)
- **Stock price forecasting**
- **Weather prediction**
- **Sales forecasting**

## Why Deep Learning for Time Series?

Traditional statistical models (like ARIMA or Exponential Smoothing) have been widely used for time series forecasting. However, these models often struggle with:

- **Complex, non-linear patterns** in the data
- **Multiple seasonalities** or trends
- **Large-scale, high-dimensional datasets**

Deep learning models, especially neural networks, have shown great promise in overcoming these limitations by learning complex patterns directly from the data.

## What is Deep Learning?

Deep learning is a subset of machine learning that uses **artificial neural networks** with multiple layers (hence "deep") to model complex relationships. Each layer transforms the data, allowing the network to learn hierarchical representations.

### Analogy

Think of deep learning as a team of detectives, where each detective (layer) specializes in finding certain clues. The first detective looks for simple clues (like lines or shapes), the next combines those into more complex patterns, and so on. By the end, the team can solve very complex mysteries!

## Neural Networks for Time Series

The most common deep learning architectures for time series forecasting include:

- **Feedforward Neural Networks (FNNs):** Basic networks that can model simple relationships.
- **Recurrent Neural Networks (RNNs):** Designed to handle sequential data by maintaining a memory of previous inputs.
- **Long Short-Term Memory (LSTM) and Gated Recurrent Unit (GRU):** Special types of RNNs that can capture long-term dependencies.
- **Temporal Convolutional Networks (TCNs):** Use convolutional layers to model temporal patterns.
- **Transformer Models:** Recently, transformers have achieved state-of-the-art results in many sequence modeling tasks, including time series.

## How Does Deep Learning Work for Time Series?

At a high level, deep learning models for time series take a sequence of past observations as input and learn to predict future values. The model automatically learns which patterns and features are important, without the need for manual feature engineering.

### Example: Forecasting Energy Consumption

Suppose we have half-hourly energy consumption data from a smart meter. A deep learning model can learn:

- Daily and weekly usage patterns
- Effects of holidays or special events
- Sudden changes in behavior

## Mathematical Representation

Given a time series $\{y_t\}_{t=1}^T$, the goal is to predict future values $y_{T+1}, y_{T+2}, \ldots, y_{T+h}$ using past observations. A deep learning model learns a function $f$ such that:

$$
\hat{y}_{T+h} = f(y_T, y_{T-1}, \ldots, y_{T-p+1}; \theta)
$$

where:
- $p$ is the number of past observations used (the "window size")
- $\theta$ are the model parameters learned during training

## Key Advantages

- **Automatic feature extraction:** Learns relevant features from raw data
- **Handles non-linearity:** Captures complex relationships
- **Scalability:** Can process large datasets efficiently

## Common Beginner Questions

**Q: Do I need a lot of data for deep learning?**  
*A: Yes, deep learning models typically require more data than traditional models to perform well.*

**Q: Are deep learning models always better?**  
*A: Not always. For simple or small datasets, traditional models may outperform deep learning. It's important to compare different approaches.*

**Q: Is deep learning a "black box"?**  
*A: Deep learning models can be less interpretable, but there are tools and techniques to help understand their predictions.*

---

In the next sections, we'll explore how to prepare time series data for deep learning, build models using the `nixtla` library, and visualize results with `plotly`.

In [4]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

unique_id,ds,start_timestamp,frequency,y,series_length,stdorToU,Acorn,Acorn_grouped,file,holidays,visibility,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary,__index_level_0__
str,list[datetime[ns]],datetime[ns],str,list[f64],i64,str,str,str,str,list[str],list[f64],list[i64],list[f64],list[f64],list[f64],list[f64],list[f64],list[str],list[str],list[f64],list[str],i64
"""MAC000002""","[2012-10-13 00:00:00, 2012-10-13 00:30:00, … 2014-02-27 23:30:00]",2012-10-13 00:00:00,"""30min""","[0.263, 0.269, … 1.2180001]",24144,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[13.08, 13.08, … 14.03]","[186, 186, … 200]","[8.78, 8.78, … 3.93]","[6.28, 6.28, … 1.61]","[1007.7, 1007.7, … 1004.62]","[7.55, 7.55, … 1.42]","[2.28, 2.28, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""clear-night"", ""clear-night"", … ""clear-night""]","[0.84, 0.84, … 0.85]","[""Clear"", ""Clear"", … ""Clear""]",0
"""MAC000246""","[2012-01-01 00:00:00, 2012-01-01 00:30:00, … 2014-02-27 23:30:00]",2012-01-01 00:00:00,"""30min""","[0.509, 0.317, … 0.223]",37872,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[12.99, 12.99, … 14.03]","[229, 229, … 200]","[12.12, 12.12, … 3.93]","[10.97, 10.97, … 1.61]","[1008.1, 1008.1, … 1004.62]","[12.12, 12.12, … 1.42]","[5.9, 5.9, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""partly-cloudy-night"", ""partly-cloudy-night"", … ""clear-night""]","[0.93, 0.93, … 0.85]","[""Mostly Cloudy"", ""Mostly Cloudy"", … ""Clear""]",1
"""MAC000450""","[2012-03-23 00:00:00, 2012-03-23 00:30:00, … 2014-02-27 23:30:00]",2012-03-23 00:00:00,"""30min""","[1.337, 1.426, … null]",33936,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[3.19, 3.19, … 14.03]","[78, 78, … 200]","[8.76, 8.76, … 3.93]","[7.25, 7.25, … 1.61]","[1027.41, 1027.41, … 1004.62]","[7.59, 7.59, … 1.42]","[2.18, 2.18, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""fog"", ""fog"", … ""clear-night""]","[0.9, 0.9, … 0.85]","[""Foggy"", ""Foggy"", … ""Clear""]",2
"""MAC001074""","[2012-05-09 00:00:00, 2012-05-09 00:30:00, … 2014-02-27 23:30:00]",2012-05-09 00:00:00,"""30min""","[0.18, 0.086, … null]",31680,"""ToU""","""ACORN-""","""ACORN-""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[10.51, 10.51, … 14.03]","[215, 215, … 200]","[11.46, 11.46, … 3.93]","[10.23, 10.23, … 1.61]","[1007.39, 1007.39, … 1004.62]","[11.46, 11.46, … 1.42]","[2.35, 2.35, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""partly-cloudy-night"", ""partly-cloudy-night"", … ""clear-night""]","[0.92, 0.92, … 0.85]","[""Partly Cloudy"", ""Partly Cloudy"", … ""Clear""]",3
"""MAC003223""","[2012-09-18 00:00:00, 2012-09-18 00:30:00, … 2014-02-27 23:30:00]",2012-09-18 00:00:00,"""30min""","[0.076, 0.079, … 0.38]",25344,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[13.44, 13.44, … 14.03]","[236, 236, … 200]","[14.06, 14.06, … 3.93]","[10.82, 10.82, … 1.61]","[1011.09, 1011.09, … 1004.62]","[14.06, 14.06, … 1.42]","[3.86, 3.86, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""clear-night"", ""clear-night"", … ""clear-night""]","[0.81, 0.81, … 0.85]","[""Clear"", ""Clear"", … ""Clear""]",4


In [5]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [6]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select([time_, id_, target_])
    .explode([time_, target_])
)
data.head()

ds,unique_id,y
datetime[ns],str,f64
2012-01-01 00:00:00,"""MAC000050""",0.175
2012-01-01 00:30:00,"""MAC000050""",0.212
2012-01-01 01:00:00,"""MAC000050""",0.313
2012-01-01 01:30:00,"""MAC000050""",0.302
2012-01-01 02:00:00,"""MAC000050""",0.257


In [7]:
selected_id = "MAC000193"
data = data.filter(id_col.eq(selected_id)).with_columns(
    target_col.forward_fill().backward_fill()
)
data.head()

ds,unique_id,y
datetime[ns],str,f64
2012-01-01 00:00:00,"""MAC000193""",0.368
2012-01-01 00:30:00,"""MAC000193""",0.386
2012-01-01 01:00:00,"""MAC000193""",0.17
2012-01-01 01:30:00,"""MAC000193""",0.021
2012-01-01 02:00:00,"""MAC000193""",0.038


# Introduction to Recurrent Neural Networks (RNNs) for Electricity Load Demand Forecasting

## What is a Recurrent Neural Network (RNN)?

A **Recurrent Neural Network (RNN)** is a type of deep learning model specifically designed to handle sequential data, such as time series. Unlike traditional neural networks, RNNs have a unique architecture that allows them to "remember" information from previous steps in the sequence, making them especially powerful for tasks where the order of data matters.

### How Does an RNN Work?

- **Memory of the Past:**  
    At each time step $t$, an RNN takes the current input $x_t$ and combines it with information from the previous time step (the "hidden state" $h_{t-1}$). This allows the network to maintain a form of memory across the sequence.
- **Mathematical Representation:**  
    The hidden state is updated as follows:
    $$
    h_t = f(W_{xh} x_t + W_{hh} h_{t-1} + b_h)
    $$
    where:
    - $x_t$ is the input at time $t$
    - $h_{t-1}$ is the hidden state from the previous time step
    - $W_{xh}$ and $W_{hh}$ are weight matrices
    - $b_h$ is a bias term
    - $f$ is a non-linear activation function (like $\tanh$ or $\text{ReLU}$)

- **Output:**  
    The output at each time step can be computed as:
    $$
    y_t = g(W_{hy} h_t + b_y)
    $$
    where $g$ is typically a linear or non-linear function.

### Analogy

Imagine reading a book, one sentence at a time. To understand the current sentence, you need to remember what happened in previous sentences. RNNs mimic this process by carrying forward information from earlier in the sequence.

---

## Why Use RNNs for Electricity Load Demand Forecasting?

Electricity load demand is a classic time series problem: the amount of electricity consumed at any given time depends on previous consumption patterns, time of day, day of the week, season, and even special events.

### Key Reasons RNNs Are Suitable:

- **Capturing Temporal Dependencies:**  
    RNNs can learn how past electricity usage influences future demand, which is crucial for accurate forecasting.
- **Handling Variable-Length Sequences:**  
    RNNs can process sequences of different lengths, making them flexible for real-world data.
- **Modeling Complex Patterns:**  
    They can capture non-linear relationships and long-term dependencies that traditional models might miss.

---

## How Does an RNN Forecast Electricity Load?

1. **Input:**  
     The RNN receives a sequence of past electricity consumption values, e.g., $[y_{t-p+1}, \ldots, y_t]$.
2. **Processing:**  
     At each time step, the RNN updates its hidden state based on the current input and its memory of previous values.
3. **Prediction:**  
     The RNN outputs a forecast for the next time step(s), such as $y_{t+1}$, $y_{t+2}$, etc.

### Example

Suppose we want to forecast the next hour's electricity demand using the past 24 hours of data. The RNN will process the sequence of 24 hourly values and learn patterns such as daily cycles, spikes during certain hours, or drops at night.

---

## Mathematical Formulation

Given a time series $\{y_t\}_{t=1}^T$, the RNN learns a function $f$ such that:
$$
\hat{y}_{T+1} = f(y_T, y_{T-1}, \ldots, y_{T-p+1}; \theta)
$$
where:
- $p$ is the number of past observations (window size)
- $\theta$ are the model parameters learned during training

---

## Common Beginner Questions

**Q: Why not use a regular neural network?**  
*A: Regular (feedforward) neural networks treat each input independently and can't capture the sequential nature of time series data. RNNs are designed to handle sequences and remember past information.*

**Q: Can RNNs handle seasonality and trends?**  
*A: Yes, RNNs can learn seasonal patterns and trends if provided with enough data. However, for very long-term dependencies, advanced variants like LSTM or GRU are often used.*

**Q: Are RNNs hard to train?**  
*A: RNNs can be more challenging to train due to issues like vanishing gradients, but modern architectures and training techniques help address these problems.*

---

In the next section, we'll see how to implement an RNN for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [10]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]

In [8]:
import logging

from neuralforecast import NeuralForecast
from neuralforecast.models import (
    LSTM,
    NHITS,
    RNN,
    MLP,
    BiTCN,
    GRU,
    NBEATS,
    Autoformer,
    TFT,
    TCN,
    DeepAR,
    DLinear,
    TSMixer,
    PatchTST,
)
from neuralforecast.losses.pytorch import MAE

logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

In [80]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    RNN(
        input_size=2 * horizon,  # Length of input sequence
        h=horizon,  # Forecast horizon
        max_steps=200,  # Number of steps to train
        scaler_type="standard",  # Type of scaler to normalize data
        encoder_hidden_size=64,  # Defines the number of hidden units of each layer of the RNN decoder
        decoder_hidden_size=64,  # Defines the number of hidden units of each layer of the RNN decoder
        encoder_n_layers=3,  # Number of layers in the RNN encoder
        decoder_layers=1,  # Number of layers in the RNN decoder
        loss=MAE(),
        valid_loss=MAE(),
        val_check_steps=10,
        early_stop_patience_steps=10,
        # callbacks=[loss_history],  # Add the loss history callback
    ),  # Defines the number of hidden units of each layer of the RNN decoder
    # LSTM(
    #     input_size=2 * horizon,
    #     h=horizon,  # Forecast horizon
    #     max_steps=200,  # Number of steps to train
    #     scaler_type="standard",  # Type of scaler to normalize data
    #     encoder_hidden_size=64,  # Defines the size of the hidden state of the LSTM
    #     decoder_hidden_size=64,
    # ),  # Defines the number of hidden units of each layer of the MLP decoder
    # MLP(
    #     input_size=2 * horizon,  # Length of input sequence
    #     h=horizon,  # Forecast horizon
    #     max_steps=200,  # Number of steps to train
    #     scaler_type="standard",  # Type of scaler to normalize data
    #     hidden_size=64,  # Defines the number of hidden units of each layer of the MLP decoder
    # ),
    # NHITS(
    #     h=horizon,  # Forecast horizon
    #     input_size=2 * horizon,  # Length of input sequence
    #     max_steps=100,  # Number of steps to train
    #     n_freq_downsample=[2, 1, 1],
    # ),  # Downsampling factors for each stack output
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
    val_size=10000,
).drop("cutoff")

Seed set to 1


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]

In [81]:
import plotly.graph_objects as go

# Extract training and validation loss trajectories from the trained RNN model
train_loss = pl.DataFrame(
    nf.models[0].train_trajectories, schema=["step", "loss"], orient="row"
).with_columns(pl.lit("train").alias("stage"))
valid_loss = pl.DataFrame(
    nf.models[0].valid_trajectories, schema=["step", "loss"], orient="row"
).with_columns(pl.lit("valid").alias("stage"))
loss = pl.concat([train_loss, valid_loss], how="vertical")


# Plot training and validation loss curves using Plotly Express
fig = px.line(
    loss,
    x="step",
    y="loss",
    color="stage",
    # labels={"step": "Training Step", "value": "Loss", "variable": "Curve"},
    title="Training and Validation Loss Curves",
)
fig.update_layout(width=800, height=400)
fig.show()

In [82]:
fig = plot_series(y_hat, y_hat)
fig.show()
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

unique_id,metric,RNN
str,str,f64
"""MAC000193""","""mae""",0.273056
"""MAC000193""","""mse""",0.273885
"""MAC000193""","""rmse""",0.52334
"""MAC000193""","""mape""",3.222816
"""MAC000193""","""smape""",0.330203
"""MAC000193""","""mase""",1.58032


# Introduction to Long Short-Term Memory (LSTM) Networks

## What is an LSTM?

A **Long Short-Term Memory (LSTM)** network is a special type of Recurrent Neural Network (RNN) designed to better capture long-term dependencies in sequential data, such as time series. While standard RNNs can "remember" information from previous steps, they often struggle with learning patterns that span many time steps due to issues like the *vanishing gradient problem*. LSTMs were specifically invented to solve this challenge.

---

## Why Do We Need LSTMs?

Imagine trying to predict electricity demand not just based on the last hour, but also considering patterns from days or even weeks ago (like weekends or holidays). Standard RNNs can quickly "forget" these distant influences. LSTMs, however, are built to remember important information for much longer periods, making them ideal for time series forecasting tasks where both short-term and long-term patterns matter.

---

## How Does an LSTM Work?

LSTMs introduce a more complex "cell" structure compared to basic RNNs. Each LSTM cell contains:

- **Cell state ($C_t$):** The memory of the network, carrying information across time steps.
- **Hidden state ($h_t$):** The output at each time step, similar to standard RNNs.
- **Gates:** Special mechanisms that control the flow of information:
    - **Forget gate ($f_t$):** Decides what information to discard from the cell state.
    - **Input gate ($i_t$):** Decides what new information to add to the cell state.
    - **Output gate ($o_t$):** Decides what information to output.

### Mathematical Representation

At each time step $t$, the LSTM updates its states as follows:

$$
\begin{aligned}
f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f) \\\\
i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i) \\\\
o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o) \\\\
\tilde{C}_t &= \tanh(W_C x_t + U_C h_{t-1} + b_C) \\\\
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \\\\
h_t &= o_t \odot \tanh(C_t)
\end{aligned}
$$

Where:
- $x_t$ is the input at time $t$
- $h_{t-1}$ is the previous hidden state
- $C_{t-1}$ is the previous cell state
- $W$, $U$, $b$ are weights and biases learned during training
- $\sigma$ is the sigmoid activation function
- $\tanh$ is the hyperbolic tangent activation function
- $\odot$ denotes element-wise multiplication

---

## Analogy

Think of the LSTM as a smart conveyor belt in a factory. The conveyor belt (cell state) carries important information down the line. At each station (time step), workers (gates) decide what to keep, what to add, and what to send out as a report (hidden state). This way, crucial information can travel a long distance without being lost or overwritten.

---

## Why Are LSTMs Powerful for Time Series Forecasting?

- **Remembering Long-Term Patterns:** LSTMs can learn both short-term fluctuations (like daily cycles) and long-term trends (like seasonal effects or gradual changes).
- **Handling Missing or Noisy Data:** The gating mechanisms help LSTMs ignore irrelevant or noisy information.
- **Flexibility:** LSTMs can be stacked into deeper networks or combined with other architectures for even greater modeling power.

---

## Common Beginner Questions

**Q: Why not just use a regular RNN?**  
*A: Regular RNNs struggle to learn long-term dependencies due to the vanishing gradient problem. LSTMs solve this by using gates to control information flow.*

**Q: Are LSTMs harder to train?**  
*A: LSTMs have more parameters than basic RNNs, so they can take longer to train, but they often achieve much better results on complex time series.*

**Q: Can LSTMs handle multiple features?**  
*A: Yes! LSTMs can process sequences with multiple input features (such as temperature, humidity, and previous energy usage) at each time step.*

---

In the next section, we'll see how to implement an LSTM for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [13]:
plot_series(y_hat, y_hat, models=["LSTM"])

# Introduction to Gated Recurrent Units (GRU)

## What is a GRU?

A **Gated Recurrent Unit (GRU)** is a type of Recurrent Neural Network (RNN) architecture designed to efficiently capture dependencies in sequential data, such as time series. GRUs were introduced as a simpler alternative to Long Short-Term Memory (LSTM) networks, aiming to solve the same problems but with fewer parameters and a more streamlined structure.

---

## Why Were GRUs Developed?

Standard RNNs struggle to learn long-term patterns due to issues like the *vanishing gradient problem*, where the influence of earlier data points fades as the sequence progresses. LSTMs addressed this with a complex gating mechanism, but GRUs simplify this further, making them faster to train and easier to implement—while still being highly effective.

---

## How Does a GRU Work?

GRUs use two main gates to control the flow of information:

- **Update Gate ($z_t$):** Decides how much of the past information to keep.
- **Reset Gate ($r_t$):** Decides how much of the past information to forget.

These gates help the GRU decide what information should be passed along the sequence and what should be updated with new input.

### Mathematical Representation

At each time step $t$, the GRU updates its hidden state as follows:

$$
\begin{aligned}
z_t &= \sigma(W_z x_t + U_z h_{t-1} + b_z) \\\\
r_t &= \sigma(W_r x_t + U_r h_{t-1} + b_r) \\\\
\tilde{h}_t &= \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h) \\\\
h_t &= (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
\end{aligned}
$$

Where:
- $x_t$ is the input at time $t$
- $h_{t-1}$ is the previous hidden state
- $z_t$ is the update gate
- $r_t$ is the reset gate
- $\tilde{h}_t$ is the candidate hidden state
- $W$, $U$, $b$ are weights and biases learned during training
- $\sigma$ is the sigmoid activation function
- $\tanh$ is the hyperbolic tangent activation function
- $\odot$ denotes element-wise multiplication

---

## Analogy

Imagine a GRU as a smart editor working on a document. The editor decides, at each sentence, whether to keep the previous content (update gate) or start fresh (reset gate). This way, the editor can efficiently manage both short-term and long-term information without getting overwhelmed.

---

## Why Use GRUs for Time Series Forecasting?

- **Efficient Memory:** GRUs can remember important patterns over long sequences, just like LSTMs, but with fewer parameters.
- **Faster Training:** Their simpler structure often leads to quicker training and less risk of overfitting.
- **Competitive Performance:** In many cases, GRUs perform as well as or even better than LSTMs, especially when data is limited or model simplicity is important.

---

## Common Beginner Questions

**Q: How are GRUs different from LSTMs?**  
*A: GRUs have a simpler structure with only two gates (update and reset), while LSTMs have three (input, forget, and output). This makes GRUs faster and easier to train.*

**Q: When should I use a GRU instead of an LSTM?**  
*A: If you want a faster, simpler model and your data doesn’t require the extra complexity of LSTMs, GRUs are a great choice.*

**Q: Can GRUs handle missing or noisy data?**  
*A: Yes, the gating mechanisms help GRUs ignore irrelevant or noisy information, making them robust for real-world time series.*

---

In the next section, we’ll see how to implement a GRU for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [14]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    GRU(
        input_size=2 * horizon,
        h=horizon,  # Forecast horizon
        max_steps=200,  # Number of steps to train
        scaler_type="standard",  # Type of scaler to normalize data
        encoder_hidden_size=64,  # Defines the size of the hidden state of the LSTM
        decoder_hidden_size=64,
    ),  # Defines the number of hidden units of each layer of the MLP decoder
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
).drop("cutoff")

Seed set to 1


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]

In [19]:
fig = plot_series(y_hat, y_hat)
fig.show()
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

unique_id,metric,GRU
str,str,f64
"""MAC000193""","""mae""",0.27645
"""MAC000193""","""mse""",0.279755
"""MAC000193""","""rmse""",0.528919
"""MAC000193""","""mape""",3.313902
"""MAC000193""","""smape""",0.350597
"""MAC000193""","""mase""",1.599962


# Introduction to N-BEATS: Neural Basis Expansion Analysis for Time Series Forecasting

## What is N-BEATS?

**N-BEATS** (Neural Basis Expansion Analysis for Time Series) is a deep learning architecture specifically designed for univariate time series forecasting. Unlike traditional statistical models or even many neural network approaches, N-BEATS is highly flexible and does not require prior knowledge about the structure of the time series (such as seasonality or trend). This makes it a powerful "black-box" model that can adapt to a wide variety of forecasting tasks.

---

## Why Was N-BEATS Developed?

Most classical time series models (like ARIMA or Exponential Smoothing) rely on strong assumptions about the data, such as linearity, stationarity, or known seasonal patterns. Even many neural network models (like RNNs or LSTMs) are designed to mimic these assumptions or require careful feature engineering.

N-BEATS was introduced to overcome these limitations by:

- **Eliminating the need for manual feature engineering**
- **Providing a generic, interpretable, and scalable neural forecasting model**
- **Achieving state-of-the-art accuracy on a wide range of forecasting benchmarks**

---

## How Does N-BEATS Work?

At its core, N-BEATS is a stack of fully connected (feedforward) neural network blocks. Each block is responsible for modeling a part of the time series and producing two outputs:

- **Backcast:** An estimate of the input time series (what the block "explains" from the past)
- **Forecast:** A prediction of future values (the actual forecast)

The model works by iteratively refining its predictions:

1. The first block tries to explain as much of the input as possible and makes an initial forecast.
2. The next block receives the residual (what's left unexplained) and tries to model that, producing its own backcast and forecast.
3. This process repeats for several blocks, and the final forecast is the sum of all block forecasts.

### Mathematical Representation

Given a window of past observations $[y_{t-p+1}, \ldots, y_t]$, each block $i$ in the stack computes:

$$
\begin{aligned}
\text{Backcast}_i, \ \text{Forecast}_i = \text{Block}_i(\text{Residual}_{i-1}) \\
\text{Residual}_i = \text{Residual}_{i-1} - \text{Backcast}_i
\end{aligned}
$$

The final forecast is:

$$
\hat{y}_{t+1:t+h} = \sum_{i=1}^{N} \text{Forecast}_i
$$

where $N$ is the number of blocks.

---

## Key Features and Advantages

- **No Recurrence or Convolution:** N-BEATS uses only fully connected layers, making it simple and fast to train.
- **Interpretable:** The backcast/forecast decomposition allows for some interpretability, especially in the "interpretable" variant of N-BEATS.
- **State-of-the-Art Performance:** N-BEATS has achieved top results on many forecasting competitions and benchmarks.
- **Versatile:** Works well for a wide range of time series, even when little is known about the underlying patterns.

---

## Limitations and Considerations

While N-BEATS is a powerful and flexible model, it's important for beginners to be aware of some potential drawbacks:

- **Primarily Univariate:** The original N-BEATS architecture is designed for univariate time series. Extensions are needed for multivariate forecasting tasks.
- **Interpretability:** Although the backcast/forecast structure offers some interpretability, N-BEATS is still less transparent than traditional statistical models.
- **Computational Resources:** Deep learning models like N-BEATS can require significant computational power and time to train, especially on large datasets.
- **Data Requirements:** Like most deep learning approaches, N-BEATS generally performs best with large amounts of historical data. For very small datasets, simpler models may be preferable.
- **Hyperparameter Tuning:** Achieving optimal performance may require careful tuning of model hyperparameters, which can be challenging for beginners.

Understanding these limitations helps you make informed decisions about when and how to use N-BEATS in your forecasting projects.

---

## Analogy

Imagine a team of expert forecasters, each specializing in explaining a different aspect of the past data. The first expert gives their best explanation and forecast, then the next expert focuses only on what the first missed, and so on. By the end, their combined forecasts provide a highly accurate prediction.

---


## Common Beginner Questions

**Q: Do I need to specify seasonality or trend for N-BEATS?**  
*A: No! N-BEATS learns these patterns directly from the data, without manual feature engineering.*

**Q: Is N-BEATS only for univariate time series?**  
*A: The original N-BEATS is designed for univariate forecasting, but extensions exist for multivariate cases.*

**Q: How does N-BEATS compare to LSTM or GRU?**  
*A: N-BEATS often outperforms recurrent models, especially when the time series structure is unknown or complex.*

---

In the next section, we'll see how to implement N-BEATS for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [ ]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    NBEATS(
        input_size=2 * horizon,
        h=horizon,  # Forecast horizon
        max_steps=200,  # Number of steps to train
        scaler_type="standard",  # Type of scaler to normalize data
    ),  # Defines the number of hidden units of each layer of the MLP decoder
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
).drop("cutoff")

Seed set to 1


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]

In [22]:
fig = plot_series(y_hat, y_hat)
fig.show()
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

unique_id,metric,NBEATS
str,str,f64
"""MAC000193""","""mae""",0.228172
"""MAC000193""","""mse""",0.184981
"""MAC000193""","""rmse""",0.430095
"""MAC000193""","""mape""",2.510689
"""MAC000193""","""smape""",0.40262
"""MAC000193""","""mase""",1.320553


# Introduction to N-HITS: Neural Hierarchical Interpolation for Time Series Forecasting

## What is N-HITS?

**N-HITS** (Neural Hierarchical Interpolation for Time Series) is a state-of-the-art deep learning architecture designed specifically for time series forecasting. It builds upon the success of earlier models like N-BEATS, introducing new mechanisms to better capture complex temporal patterns, such as multiple seasonalities and long-term trends, which are common in real-world data like electricity demand.

---

## Why Was N-HITS Developed?

Traditional forecasting models and even many neural network approaches can struggle with:

- **Capturing multiple seasonalities** (e.g., daily and weekly cycles)
- **Modeling long-term dependencies**
- **Efficiently handling large-scale datasets**

N-HITS was developed to address these challenges by:

- Using a hierarchical structure to model different temporal patterns at various resolutions
- Employing interpolation techniques to efficiently combine information from different time scales
- Achieving high accuracy on a wide range of forecasting benchmarks

---

## How Does N-HITS Work?

At its core, N-HITS is composed of several **blocks**, each responsible for modeling a specific aspect of the time series. Each block operates at a different temporal resolution, allowing the model to learn both short-term and long-term patterns.

### Key Components:

- **Hierarchical Blocks:**  
    Each block focuses on a different time scale (e.g., hourly, daily, weekly), capturing patterns unique to that resolution.

- **Interpolation:**  
    The outputs from each block are combined using interpolation, allowing the model to reconstruct the forecast at the original resolution.

- **Residual Learning:**  
    Like N-BEATS, each block tries to explain the part of the time series not captured by previous blocks, refining the forecast step by step.

### Mathematical Representation

Given a window of past observations $[y_{t-p+1}, \ldots, y_t]$, each block $i$ produces a forecast $\hat{y}^{(i)}_{t+1:t+h}$ at its own resolution. The final forecast is the sum of all block forecasts, interpolated to the target resolution:

$$
\hat{y}_{t+1:t+h} = \sum_{i=1}^{N} \text{Interpolate}(\hat{y}^{(i)}_{t+1:t+h})
$$

where $N$ is the number of blocks.

---

## Why Is N-HITS Powerful for Time Series Forecasting?

- **Captures Multiple Seasonalities:**  
    By modeling at different resolutions, N-HITS can learn daily, weekly, and even yearly patterns simultaneously.

- **Handles Long-Term Dependencies:**  
    The hierarchical structure allows the model to remember and utilize information from far back in the time series.

- **Scalable and Efficient:**  
    N-HITS is designed to work well with large datasets and can be trained efficiently on modern hardware.

---

## Limitations and Considerations

While N-HITS is a powerful and flexible model, it's important for beginners to be aware of some potential drawbacks:

- **Primarily Univariate:**  
    The standard N-HITS architecture is mainly designed for univariate time series. Extensions are needed for multivariate forecasting tasks.

- **Interpretability:**  
    Like many deep learning models, N-HITS can be considered a "black box," making it harder to interpret compared to traditional statistical models.

- **Computational Resources:**  
    Training N-HITS can require significant computational power and time, especially on large datasets.

- **Data Requirements:**  
    N-HITS generally performs best with large amounts of historical data. For very small datasets, simpler models may be preferable.

- **Hyperparameter Tuning:**  
    Achieving optimal performance may require careful tuning of model hyperparameters, which can be challenging for beginners.

Understanding these limitations helps you make informed decisions about when and how to use N-HITS in your forecasting projects.

---

## Analogy

Imagine forecasting electricity demand with a team of specialists: one focuses on daily patterns, another on weekly cycles, and another on long-term trends. Each specialist makes their prediction, and then their insights are combined to produce the most accurate forecast possible. N-HITS works in a similar way, using neural networks to learn from different time scales and merge their predictions.

---

## Common Beginner Questions

**Q: How is N-HITS different from N-BEATS?**  
*A: While both use block-based architectures, N-HITS introduces hierarchical blocks and interpolation, allowing it to better capture multiple seasonalities and long-term patterns.*

**Q: Do I need to specify the seasonalities in advance?**  
*A: No! N-HITS learns relevant patterns directly from the data, without manual feature engineering.*

**Q: Is N-HITS only for univariate time series?**  
*A: N-HITS is primarily designed for univariate forecasting, but extensions exist for multivariate cases.*

---

In the next section, we’ll see how to implement N-HITS for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [ ]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    NHITS(
        h=horizon,  # Forecast horizon (number of future steps to predict)
        input_size=2 * horizon,  # Length of input sequence (lookback window)
        max_steps=200,  # Number of training steps (epochs)
        scaler_type="standard",  # Normalize the data for stable training
        n_freq_downsample=[
            2,
            1,
            1,
        ],  # Downsampling factors for each block (captures multiple seasonalities)
    )
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
).drop("cutoff")

fig = plot_series(y_hat, y_hat)
fig.show()
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

Seed set to 1


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]

unique_id,metric,NHITS
str,str,f64
"""MAC000193""","""mae""",0.217546
"""MAC000193""","""mse""",0.182669
"""MAC000193""","""rmse""",0.427398
"""MAC000193""","""mape""",1.969343
"""MAC000193""","""smape""",0.417023
"""MAC000193""","""mase""",1.259053


# Introduction to Autoformer: Advanced Deep Learning for Time Series Forecasting

## What is Autoformer?

**Autoformer** is a cutting-edge deep learning architecture specifically designed for long-term time series forecasting. It builds upon the success of transformer models, which have revolutionized natural language processing, and adapts them to address the unique challenges of time series data—such as capturing long-range dependencies, handling multiple seasonalities, and efficiently modeling complex temporal patterns.

---

## Why Was Autoformer Developed?

Traditional models (like ARIMA or Exponential Smoothing) and even many neural network approaches (like RNNs or LSTMs) can struggle with:

- **Long-term forecasting:** Predicting far into the future, especially when patterns repeat over long periods.
- **Multiple seasonalities:** Handling data with several overlapping cycles (e.g., daily, weekly, yearly).
- **Scalability:** Efficiently processing large-scale time series data.

Autoformer was developed to overcome these limitations by introducing novel mechanisms that make it highly effective for both short- and long-term forecasting tasks.

---

## How Does Autoformer Work?

Autoformer is based on the **transformer** architecture, which uses self-attention mechanisms to model relationships between all points in a sequence. However, Autoformer introduces two key innovations tailored for time series:

1. **Series Decomposition Block:**  
    - Autoformer explicitly decomposes the input time series into **trend** and **seasonal** components.
    - This helps the model focus on learning each part separately, improving accuracy and interpretability.

2. **Auto-Correlation Mechanism:**  
    - Instead of standard self-attention, Autoformer uses an **auto-correlation mechanism** to efficiently capture repeating patterns and dependencies over long horizons.
    - This allows the model to identify and leverage periodicities in the data, which is crucial for time series forecasting.

### Mathematical Representation

Given a time series $\{y_t\}_{t=1}^T$, Autoformer learns to predict future values by decomposing the sequence:

$$
y_t = \text{Trend}_t + \text{Seasonal}_t
$$

The model then uses auto-correlation to find similar patterns in the past and projects them into the future, enabling accurate long-term forecasts.

---

## Analogy

Imagine forecasting electricity demand by first separating the overall upward or downward trend (like a rising or falling tide) from the regular daily or weekly cycles (like waves). Then, you look back in time to find similar patterns and use them to predict what will happen next. Autoformer automates this process using deep learning.

---

## Why Is Autoformer Powerful for Time Series Forecasting?

- **Captures Long-Term Dependencies:**  
  The auto-correlation mechanism allows Autoformer to model relationships across long time spans, making it ideal for long-range forecasting.

- **Handles Multiple Seasonalities:**  
  By decomposing the series, Autoformer can learn and combine multiple repeating patterns.

- **Efficient and Scalable:**  
  Autoformer is designed to process large datasets efficiently, making it suitable for industrial-scale forecasting tasks.

- **State-of-the-Art Performance:**  
  Autoformer has achieved top results on many public time series forecasting benchmarks.

---

## Common Beginner Questions

**Q: How is Autoformer different from a regular Transformer?**  
*A: Autoformer introduces series decomposition and auto-correlation mechanisms, making it more suitable for time series data than standard transformers.*

**Q: Do I need to manually specify trends or seasonalities?**  
*A: No! Autoformer learns to decompose and model these components automatically from the data.*

**Q: Is Autoformer only for univariate time series?**  
*A: While the original Autoformer is designed for univariate forecasting, extensions and adaptations exist for multivariate cases.*

---

In the next section, we’ll see how to implement Autoformer for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [24]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    Autoformer(
        input_size=2 * horizon,  # Length of input sequence (lookback window)
        h=horizon,  # Forecast horizon (number of future steps to predict)
        max_steps=200,  # Number of training steps (epochs)
        scaler_type="standard",  # Normalize the data for stable training
    )
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
).drop("cutoff")

fig = plot_series(y_hat, y_hat)
fig.show()
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

Seed set to 1


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]

unique_id,metric,Autoformer
str,str,f64
"""MAC000193""","""mae""",0.344669
"""MAC000193""","""mse""",0.268659
"""MAC000193""","""rmse""",0.518324
"""MAC000193""","""mape""",5.0772
"""MAC000193""","""smape""",0.583746
"""MAC000193""","""mase""",1.994779


# Introduction to DeepAR: Probabilistic Forecasting with Deep Learning

## What is DeepAR?

**DeepAR** is a deep learning-based time series forecasting model developed by Amazon. Unlike traditional models that predict a single future value, DeepAR is designed for **probabilistic forecasting**—it predicts a full probability distribution for future values, not just point estimates. This is especially useful in real-world applications where understanding uncertainty is as important as the forecast itself.

---

## Why Was DeepAR Developed?

Traditional time series models (like ARIMA or Exponential Smoothing) and even many neural network models typically:

- Predict only the expected (mean) value for each future time step.
- Struggle to scale efficiently when forecasting thousands of related time series (such as energy usage for many households).

DeepAR was developed to address these challenges by:

- Leveraging **recurrent neural networks (RNNs)** to model sequential dependencies.
- Sharing information across multiple time series to improve accuracy, especially for series with limited historical data.
- Providing **probabilistic forecasts** by modeling the entire distribution of possible future outcomes.

---

## How Does DeepAR Work?

At its core, DeepAR uses an RNN (such as LSTM or GRU) to process historical observations and generate forecasts. The key innovations are:

- **Likelihood-based Training:**  
    DeepAR assumes the target variable follows a specific probability distribution (e.g., Gaussian, Negative Binomial). The model is trained to maximize the likelihood of the observed data under this distribution.

- **Probabilistic Output:**  
    Instead of outputting a single value, DeepAR predicts the parameters of the chosen distribution for each future time step. This allows the model to generate **prediction intervals** and quantify uncertainty.

- **Global Modeling:**  
    DeepAR can be trained on many related time series simultaneously, learning shared patterns and improving forecasts for all series.

### Mathematical Representation

Given a time series $\{y_t\}_{t=1}^T$, DeepAR models the conditional distribution of future values:

$$
p(y_{T+1}, \ldots, y_{T+h} \mid y_1, \ldots, y_T; \theta)
$$

where $\theta$ are the model parameters learned during training.

At each time step, the RNN outputs the parameters of a probability distribution (e.g., mean and variance for a Gaussian), and the model is trained to maximize the likelihood of the observed data.

---

## Analogy

Imagine forecasting electricity demand not just as a single "best guess," but as a range of possible outcomes—like a weather forecast that gives you a probability of rain. DeepAR is like a smart forecaster that learns from many households at once, understands uncertainty, and tells you not just what is likely to happen, but also how confident it is in that prediction.

---

## Why Use DeepAR for Time Series Forecasting?

- **Probabilistic Forecasts:**  
    DeepAR provides full predictive distributions, allowing you to compute prediction intervals and assess risk.

- **Scalability:**  
    It can handle thousands of related time series efficiently, making it ideal for large-scale applications.

- **Handles Complex Patterns:**  
    By using RNNs, DeepAR can capture non-linear dependencies, seasonality, and trends.

- **Data Efficiency:**  
    Sharing information across series helps improve forecasts, especially for series with limited history.

---

## Common Beginner Questions

**Q: What does "probabilistic forecasting" mean?**  
*A: Instead of predicting a single value, probabilistic forecasting predicts a range of possible values and their likelihoods. This helps you understand uncertainty and make better decisions.*

**Q: Can DeepAR handle missing data?**  
*A: Yes, DeepAR can handle missing values and irregular time series by using masking and imputation techniques during training.*

**Q: Is DeepAR only for univariate time series?**  
*A: DeepAR is primarily designed for univariate series, but it can incorporate covariates (additional features) to improve forecasts.*

---

In the next section, we'll see how to implement DeepAR for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [26]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    DeepAR(
        input_size=2 * horizon,  # Length of input sequence (lookback window)
        h=horizon,  # Forecast horizon (number of future steps to predict)
        max_steps=200,  # Number of training steps (epochs)
        scaler_type="standard",  # Normalize the data for stable training
        accelerator="cpu",
    )
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
).drop("cutoff")

fig = plot_series(y_hat, y_hat)
fig.show()
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

Seed set to 1


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]

unique_id,metric,DeepAR,DeepAR-median
str,str,f64,f64
"""MAC000193""","""mae""",0.332912,0.331576
"""MAC000193""","""mse""",0.291053,0.289446
"""MAC000193""","""rmse""",0.539493,0.538002
"""MAC000193""","""mape""",5.320848,5.305327
"""MAC000193""","""smape""",0.54725,0.545873
"""MAC000193""","""mase""",1.926739,1.919006


# Introduction to PatchTST: Patch-Based Transformer for Time Series Forecasting

## What is PatchTST?

**PatchTST** (Patch-based Time Series Transformer) is a modern deep learning architecture designed specifically for time series forecasting. It adapts the powerful transformer model—originally developed for natural language processing—to better handle the unique characteristics of time series data. PatchTST achieves this by dividing the time series into small, fixed-length segments called "patches," and then learning relationships between these patches using self-attention mechanisms.

---

## Why Was PatchTST Developed?

While transformers have revolutionized sequence modeling in fields like text and images, their direct application to time series forecasting faces several challenges:

- **Long Sequences:** Time series data often consists of long sequences, making it computationally expensive for standard transformers.
- **Local Patterns:** Many important patterns in time series (like daily or weekly cycles) are local and may be missed by models that treat each time step independently.
- **Scalability:** Efficiently modeling both short-term and long-term dependencies is crucial for accurate forecasting.

PatchTST was developed to address these issues by:

- Breaking the time series into patches, allowing the model to focus on local patterns within each patch.
- Using transformer blocks to capture global dependencies between patches.
- Improving computational efficiency and scalability for long time series.

---

## How Does PatchTST Work?

The core idea of PatchTST is to treat a time series as a sequence of patches, similar to how Vision Transformers (ViT) process images as patches. Here’s how it works:

1. **Patch Extraction:**  
    The input time series is divided into overlapping or non-overlapping patches (small segments of consecutive time steps).

2. **Embedding:**  
    Each patch is embedded into a vector representation, capturing local temporal patterns.

3. **Transformer Encoder:**  
    The sequence of patch embeddings is processed by a transformer encoder, which uses self-attention to learn relationships between patches—capturing both local and global dependencies.

4. **Forecasting:**  
    The output of the transformer is used to predict future values of the time series.

### Mathematical Representation

Given a time series $\{y_t\}_{t=1}^T$, PatchTST divides it into $N$ patches:

$$
\text{Patches} = \{ [y_{1}, \ldots, y_{p}], [y_{p+1}, \ldots, y_{2p}], \ldots \}
$$

where $p$ is the patch length.

Each patch is embedded and passed through the transformer encoder, which outputs a forecast for the next $h$ time steps:

$$
\hat{y}_{T+1:T+h} = f(\text{Patches}; \theta)
$$

where $f$ is the PatchTST model and $\theta$ are the learned parameters.

---

## Analogy

Imagine reading a long novel by focusing on one paragraph (patch) at a time, understanding its meaning, and then connecting the ideas across paragraphs to grasp the whole story. PatchTST does something similar for time series: it learns from small segments and then connects them to make accurate predictions.

---

## Why Use PatchTST for Time Series Forecasting?

- **Captures Local and Global Patterns:**  
  By working with patches, PatchTST can learn both short-term (local) and long-term (global) dependencies.

- **Efficient for Long Sequences:**  
  Processing patches reduces the computational burden compared to treating every time step separately.

- **State-of-the-Art Performance:**  
  PatchTST has achieved top results on many time series forecasting benchmarks, often outperforming traditional and other deep learning models.

- **Flexible and Scalable:**  
  Suitable for univariate and multivariate time series, and can handle large datasets efficiently.

---

## Common Beginner Questions

**Q: How is PatchTST different from a regular Transformer?**  
*A: PatchTST divides the time series into patches before applying the transformer, making it more efficient and better at capturing local patterns.*

**Q: Do I need to choose the patch size?**  
*A: Yes, the patch size is a hyperparameter. It should be chosen based on the nature of your data and the patterns you want to capture.*

**Q: Can PatchTST handle multiple time series at once?**  
*A: Yes, PatchTST can be extended to multivariate time series and can process multiple series in parallel.*

---

In the next section, we’ll see how to implement PatchTST for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [29]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    PatchTST(
        input_size=2 * horizon,  # Length of input sequence (lookback window)
        h=horizon,  # Forecast horizon (number of future steps to predict)
        max_steps=200,  # Number of training steps (epochs)
        scaler_type="standard",  # Normalize the data for stable training
    )
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
).drop("cutoff")

fig = plot_series(y_hat, y_hat)
fig.show()
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

Seed set to 1


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]

unique_id,metric,PatchTST
str,str,f64
"""MAC000193""","""mae""",0.267144
"""MAC000193""","""mse""",0.18671
"""MAC000193""","""rmse""",0.4321
"""MAC000193""","""mape""",4.351931
"""MAC000193""","""smape""",0.435939
"""MAC000193""","""mase""",1.546102


# Introduction to Temporal Fusion Transformers (TFT)

## What is TFT?

**Temporal Fusion Transformer (TFT)** is a state-of-the-art deep learning architecture designed specifically for time series forecasting. TFT combines the strengths of recurrent neural networks (RNNs), attention mechanisms, and interpretable neural network components to handle complex, real-world forecasting tasks. It is especially powerful for datasets with multiple variables (features), missing values, and both static and time-varying covariates.

---

## Why Was TFT Developed?

Traditional time series models and even many deep learning approaches often struggle with:

- **Handling multiple input features:** Many models are designed for univariate series and require extensive feature engineering for multivariate data.
- **Capturing both short-term and long-term dependencies:** Some models focus on local patterns, while others capture global trends, but few do both well.
- **Interpretability:** Deep learning models are often seen as "black boxes," making it hard to understand their predictions.

TFT was developed to address these challenges by:

- Integrating attention mechanisms for both variable selection and temporal relationships.
- Combining static (unchanging) and dynamic (time-varying) features.
- Providing built-in interpretability, so you can see which features and time steps influenced the forecast.

---

## How Does TFT Work?

TFT’s architecture is composed of several key components:

1. **Variable Selection Networks:**  
    At each time step, TFT learns which input features (variables) are most relevant for forecasting, allowing it to focus on the most important information.

2. **Gated Residual Networks (GRN):**  
    These networks help the model learn complex, non-linear relationships between variables, while also allowing information to flow easily through the network.

3. **Temporal Attention:**  
    TFT uses attention mechanisms to focus on the most relevant time steps in the past, capturing both short-term and long-term dependencies.

4. **Static Covariate Encoders:**  
    TFT can incorporate static features (like customer type or region) that do not change over time, improving forecast accuracy.

5. **Interpretable Outputs:**  
    TFT provides attention weights and variable importance scores, making it easier to understand how the model makes predictions.

### Mathematical Representation

Given a time series $\{y_t\}_{t=1}^T$ with associated features $\mathbf{x}_t$, TFT learns a function $f$ such that:

$$
\hat{y}_{T+1:T+h} = f(y_{1:T}, \mathbf{x}_{1:T}, \mathbf{s}; \theta)
$$

where:
- $\mathbf{x}_t$ are time-varying features at time $t$
- $\mathbf{s}$ are static features
- $\theta$ are the model parameters

---

## Analogy

Imagine forecasting electricity demand for a household. You have access to not just past usage, but also weather, holidays, and household characteristics. TFT acts like a team of expert analysts: some focus on the most important features, others look for patterns over different time periods, and together they combine their insights to make an accurate and interpretable forecast.

---

## Why Use TFT for Time Series Forecasting?

- **Handles Complex Data:**  
  TFT can process multivariate time series with static and dynamic features, missing values, and irregular time steps.

- **Captures Multiple Patterns:**  
  The attention mechanism allows TFT to learn both short-term and long-term dependencies.

- **Interpretable:**  
  TFT provides insights into which features and time steps are most important for each prediction.

- **State-of-the-Art Performance:**  
  TFT has achieved top results on many real-world forecasting benchmarks.

---

## Common Beginner Questions

**Q: Can TFT handle missing data?**  
*A: Yes, TFT is designed to handle missing values and irregular time series.*

**Q: Is TFT only for univariate time series?**  
*A: No, TFT excels at multivariate forecasting, using both static and time-varying features.*

**Q: Is TFT easy to interpret?**  
*A: Yes! TFT provides variable importance and attention weights, helping you understand its predictions.*

---

In the next section, we’ll see how to implement TFT for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [30]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    TFT(
        input_size=2 * horizon,  # Length of input sequence (lookback window)
        h=horizon,  # Forecast horizon (number of future steps to predict)
        max_steps=200,  # Number of training steps (epochs)
        scaler_type="standard",  # Normalize the data for stable training
    )
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
).drop("cutoff")

fig = plot_series(y_hat, y_hat)
fig.show()
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

Seed set to 1


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]

unique_id,metric,TFT
str,str,f64
"""MAC000193""","""mae""",0.284568
"""MAC000193""","""mse""",0.205512
"""MAC000193""","""rmse""",0.453335
"""MAC000193""","""mape""",4.086997
"""MAC000193""","""smape""",0.565233
"""MAC000193""","""mase""",1.646946


# Introduction to BiTCN: Bidirectional Temporal Convolutional Networks for Time Series Forecasting

## What is BiTCN?

**BiTCN** stands for **Bidirectional Temporal Convolutional Network**. It is a deep learning architecture designed specifically for time series forecasting. BiTCN builds on the concept of Temporal Convolutional Networks (TCNs), which use convolutional layers to model sequential data, but adds a **bidirectional** component—allowing the model to learn from both past and future contexts within a window of data.

---

## Why Was BiTCN Developed?

Traditional models like RNNs and LSTMs process time series data sequentially, step by step. While powerful, these models can be slow to train and may struggle to capture long-range dependencies efficiently. TCNs address these issues by using convolutional layers, which can process sequences in parallel and capture patterns over long time spans.

However, standard TCNs are **unidirectional**—they only use past information to predict the future. BiTCN improves upon this by also considering "future" context within the input window, making it especially effective for tasks where both past and near-future patterns are informative.

---

## How Does BiTCN Work?

- **Convolutional Layers:**  
    BiTCN applies 1D convolutions across the time dimension of the input sequence. This allows the model to learn local and global temporal patterns efficiently.

- **Bidirectionality:**  
    Instead of only looking backward in time, BiTCN processes the input sequence in both forward and backward directions. This means, for each point in the sequence, the model can use information from both earlier and later time steps within the input window.

- **Parallel Processing:**  
    Unlike RNNs, which process data sequentially, BiTCN can process entire sequences in parallel, making training and inference much faster.

### Mathematical Representation

Given an input sequence $[y_{t-p+1}, \ldots, y_t]$, BiTCN applies convolutional filters in both directions:

- **Forward TCN:** Processes the sequence from $y_{t-p+1}$ to $y_t$.
- **Backward TCN:** Processes the sequence from $y_t$ to $y_{t-p+1}$.

The outputs from both directions are combined (e.g., concatenated or added) to produce the final representation, which is then used to forecast future values.

---

## Analogy

Imagine you’re trying to predict the next scene in a movie. Instead of only remembering what happened before, you also get a sneak peek at a few scenes ahead. By combining clues from both the past and the near future, you can make a much better prediction about what happens next. BiTCN does something similar for time series data.

---

## Why Use BiTCN for Time Series Forecasting?

- **Captures Local and Global Patterns:**  
    Convolutional layers can learn both short-term and long-term dependencies.

- **Bidirectional Context:**  
    By looking at both past and near-future data within the input window, BiTCN can make more accurate forecasts.

- **Efficient Training:**  
    Parallel processing makes BiTCN faster to train than sequential models like RNNs.

- **Robust to Sequence Length:**  
    BiTCN can handle long input sequences without the vanishing gradient problems common in RNNs.

---

## Common Beginner Questions

**Q: How is BiTCN different from a regular TCN?**  
*A: BiTCN processes the input sequence in both forward and backward directions, while a regular TCN only looks backward (past context).*

**Q: Can BiTCN be used for real-time forecasting?**  
*A: For real-time (online) forecasting, only past data is available, so standard TCNs are used. BiTCN is most useful when you have access to a window of data and want to leverage both past and future context within that window.*

**Q: Is BiTCN suitable for multivariate time series?**  
*A: Yes! BiTCN can process multiple features at each time step, making it flexible for both univariate and multivariate forecasting tasks.*

---

In the next section, we’ll see how to implement BiTCN for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [31]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    BiTCN(
        input_size=2 * horizon,  # Length of input sequence (lookback window)
        h=horizon,  # Forecast horizon (number of future steps to predict)
        max_steps=200,  # Number of training steps (epochs)
        scaler_type="standard",  # Normalize the data for stable training
    )
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
).drop("cutoff")

fig = plot_series(y_hat, y_hat)
fig.show()
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

Seed set to 1


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]

unique_id,metric,BiTCN
str,str,f64
"""MAC000193""","""mae""",0.268711
"""MAC000193""","""mse""",0.210451
"""MAC000193""","""rmse""",0.458749
"""MAC000193""","""mape""",4.029977
"""MAC000193""","""smape""",0.436964
"""MAC000193""","""mase""",1.555169


# Introduction to TSMixer: MLP-Based Deep Learning for Time Series Forecasting

## What is TSMixer?

**TSMixer** is a modern deep learning architecture designed specifically for time series forecasting. Unlike traditional models that rely on recurrent (RNN) or convolutional (CNN) layers, TSMixer is built entirely from **multi-layer perceptrons (MLPs)**. This makes it both simple and highly efficient, while still being powerful enough to capture complex temporal patterns in time series data.

---

## Why Was TSMixer Developed?

Most deep learning models for time series—such as RNNs, LSTMs, and Transformers—are designed to handle sequential data by explicitly modeling the order of time steps. However, these models can be:

- **Computationally intensive:** Especially for long sequences.
- **Difficult to scale:** When dealing with large datasets or many time series.
- **Complex to tune:** Due to many architectural choices.

TSMixer was developed to address these challenges by:

- Using only MLP layers, which are fast and easy to train.
- Mixing information across both the **time** and **feature** dimensions, allowing the model to learn dependencies without explicit sequence modeling.
- Achieving competitive performance with much simpler architecture.

---

## How Does TSMixer Work?

TSMixer processes time series data by alternating between two types of MLP blocks:

1. **Time-Mixing MLP:**  
    - Learns relationships across different time steps for each feature.
    - Helps the model understand how past values influence the future.

2. **Feature-Mixing MLP:**  
    - Learns relationships between different features (variables) at each time step.
    - Useful for multivariate time series where multiple variables interact.

By stacking these blocks, TSMixer can capture both temporal and cross-feature dependencies, all without using recurrence or attention mechanisms.

### Mathematical Representation

Given a multivariate time series $X \in \mathbb{R}^{T \times F}$, where $T$ is the number of time steps and $F$ is the number of features:

- **Time-mixing:**  
  For each feature $f$, apply an MLP across the time dimension:
  $$
  X^{(t)}_f = \text{MLP}_\text{time}(X_{:, f})
  $$

- **Feature-mixing:**  
  For each time step $t$, apply an MLP across the feature dimension:
  $$
  X^{(f)}_t = \text{MLP}_\text{feature}(X_{t, :})
  $$

These steps are repeated in sequence, allowing the model to "mix" information across both axes.

---

## Analogy

Imagine you’re analyzing a spreadsheet of electricity usage, where each row is a time step and each column is a different measurement (like temperature, humidity, or previous usage). TSMixer first looks down each column to find patterns over time, then looks across each row to see how the variables interact at each moment. By repeating this process, it builds a rich understanding of the data.

---

## Why Use TSMixer for Time Series Forecasting?

- **Simplicity:**  
  The architecture is easy to implement and tune, making it accessible for beginners.

- **Efficiency:**  
  MLPs are computationally lightweight, enabling fast training and inference.

- **Competitive Accuracy:**  
  Despite its simplicity, TSMixer achieves results comparable to more complex models on many benchmarks.

- **Flexible:**  
  Works well for both univariate and multivariate time series.

---

## Common Beginner Questions

**Q: How is TSMixer different from RNNs or Transformers?**  
*A: TSMixer uses only MLP layers, without recurrence or attention. It mixes information across time and features using simple feedforward networks.*

**Q: Can TSMixer handle multiple variables?**  
*A: Yes! The feature-mixing blocks are designed to learn interactions between multiple variables at each time step.*

**Q: Is TSMixer suitable for long time series?**  
*A: Yes, TSMixer is efficient and can handle long sequences, but the optimal performance may depend on the specific dataset and task.*

---

In the next section, we’ll see how to implement TSMixer for electricity load forecasting using the `nixtla` library and visualize the results with `plotly`.

In [33]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    TSMixer(
        input_size=2 * horizon,  # Length of input sequence (lookback window)
        h=horizon,  # Forecast horizon (number of future steps to predict)
        max_steps=200,  # Number of training steps (epochs)
        scaler_type="standard",  # Normalize the data for stable training
        n_series=1,
    )
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
).drop("cutoff")

fig = plot_series(y_hat, y_hat)
fig.show()
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

Seed set to 1


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]

unique_id,metric,TSMixer
str,str,f64
"""MAC000193""","""mae""",0.284585
"""MAC000193""","""mse""",0.211453
"""MAC000193""","""rmse""",0.45984
"""MAC000193""","""mape""",4.589488
"""MAC000193""","""smape""",0.486165
"""MAC000193""","""mase""",1.647041
